# Chapter 8: Feature Engineering for Soccer Analytics

This notebook is the canonical runnable companion for Chapter 8. The examples use synthetic StatsBomb-style match and event data so the full chapter can run top to bottom. The flow mirrors the chapter's four-layer structure: performance, tactical, contextual, and ranking features.


## Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)


## Synthetic Match and Event Data


In [ ]:
teams = [
    "United States Women's",
    "Netherlands Women's",
    "England Women's",
    "Sweden Women's",
    "Germany Women's",
    "France Women's",
    "Spain Women's",
    "Australia Women's",
]

team_strength = {
    "United States Women's": 0.92,
    "Netherlands Women's": 0.85,
    "England Women's": 0.82,
    "Sweden Women's": 0.78,
    "Germany Women's": 0.80,
    "France Women's": 0.79,
    "Spain Women's": 0.77,
    "Australia Women's": 0.72,
}

def build_matches_df():
    matches = []
    base_date = pd.Timestamp('2019-06-01')
    match_id = 69000
    day_offset = 0

    for home_index, team_a in enumerate(teams):
        for away_index, team_b in enumerate(teams[home_index + 1:], start=home_index + 1):
            if {team_a, team_b} == {
                "United States Women's",
                "Netherlands Women's",
            }:
                continue

            home_team, away_team = (
                (team_a, team_b)
                if (home_index + away_index) % 2 == 0
                else (team_b, team_a)
            )

            match_id += 1
            home_strength = team_strength[home_team] + 0.05
            away_strength = team_strength[away_team]

            home_rate = np.clip(
                0.6 + 1.7 * home_strength - 0.7 * away_strength,
                0.3,
                None,
            )
            away_rate = np.clip(
                0.4 + 1.5 * away_strength - 0.5 * home_strength,
                0.2,
                None,
            )

            matches.append({
                'match_id': match_id,
                'match_date': base_date + pd.Timedelta(days=day_offset),
                'home_team': home_team,
                'away_team': away_team,
                'home_score': int(np.random.poisson(home_rate)),
                'away_score': int(np.random.poisson(away_rate)),
            })
            day_offset += 3

    matches.append({
        'match_id': 69321,
        'match_date': base_date + pd.Timedelta(days=day_offset + 4),
        'home_team': "United States Women's",
        'away_team': "Netherlands Women's",
        'home_score': 2,
        'away_score': 0,
    })

    return pd.DataFrame(matches).sort_values('match_date').reset_index(drop=True)

def build_events_df(matches_df):
    event_rows = []

    for _, match in matches_df.iterrows():
        match_id = match['match_id']
        team_pairs = [
            (match['home_team'], match['away_team'], int(match['home_score'])),
            (match['away_team'], match['home_team'], int(match['away_score'])),
        ]

        for team, opponent, goals in team_pairs:
            strength = team_strength[team]
            opponent_strength = team_strength[opponent]

            num_passes = max(260, int(np.random.normal(320 + 120 * strength, 25)))
            num_shots = max(goals + 4, int(np.random.poisson(6 + 5 * strength)))
            num_duels = max(8, int(np.random.poisson(10 + 6 * (1 - strength))))
            num_interceptions = max(
                4,
                int(np.random.poisson(6 + 4 * (1 - strength) + 1.5 * opponent_strength)),
            )
            num_recoveries = max(5, int(np.random.poisson(7 + 5 * (1 - strength))))

            assist_count = min(goals, max(1, num_passes // 90)) if goals else 0
            assist_indices = (
                set(np.random.choice(num_passes, size=assist_count, replace=False))
                if assist_count
                else set()
            )
            pass_types = np.random.choice(
                np.array([None, 'Corner', 'Free Kick'], dtype=object),
                size=num_passes,
                p=[0.88, 0.07, 0.05],
            )

            for pass_index in range(num_passes):
                x_start = float(np.random.uniform(5, 95))
                y_start = float(np.random.uniform(0, 80))
                x_end = float(
                    np.clip(x_start + np.random.normal(12 + 6 * strength, 9), 0, 120)
                )
                y_end = float(np.clip(y_start + np.random.normal(0, 12), 0, 80))
                distance = float(np.hypot(x_end - x_start, y_end - y_start))
                success_prob = np.clip(
                    0.74 + 0.15 * strength - 0.0035 * distance,
                    0.55,
                    0.95,
                )

                event_rows.append({
                    'match_id': match_id,
                    'team': team,
                    'type': 'Pass',
                    'location': [round(x_start, 2), round(y_start, 2)],
                    'pass_end_location': [round(x_end, 2), round(y_end, 2)],
                    'pass_outcome': None if np.random.rand() < success_prob else 'Incomplete',
                    'pass_goal_assist': int(pass_index in assist_indices),
                    'pass_type': pass_types[pass_index],
                    'shot_statsbomb_xg': np.nan,
                    'shot_outcome': None,
                    'play_pattern': None,
                })

            goal_count = min(goals, num_shots)
            goal_indices = (
                set(np.random.choice(num_shots, size=goal_count, replace=False))
                if goal_count
                else set()
            )
            shot_patterns = np.random.choice(
                ['Open Play', 'From Corner', 'From Free Kick', 'From Penalty'],
                size=num_shots,
                p=[0.72, 0.14, 0.09, 0.05],
            )

            for shot_index in range(num_shots):
                x_coord = float(np.random.uniform(86, 118))
                y_coord = float(np.random.uniform(18, 62))
                xg_value = float(
                    np.clip(
                        np.random.beta(2 + 2 * strength, 8) * 0.8,
                        0.02,
                        0.75,
                    )
                )
                play_pattern = shot_patterns[shot_index]

                event_rows.append({
                    'match_id': match_id,
                    'team': team,
                    'type': 'Shot',
                    'location': [round(x_coord, 2), round(y_coord, 2)],
                    'pass_end_location': None,
                    'pass_outcome': None,
                    'pass_goal_assist': 0,
                    'pass_type': None,
                    'shot_statsbomb_xg': round(xg_value, 3),
                    'shot_outcome': 'Goal'
                    if shot_index in goal_indices
                    else np.random.choice(['Saved', 'Off T', 'Blocked'], p=[0.55, 0.25, 0.20]),
                    'play_pattern': play_pattern,
                })

                if play_pattern == 'From Penalty':
                    event_rows.append({
                        'match_id': match_id,
                        'team': team,
                        'type': 'Penalty',
                        'location': [round(x_coord, 2), round(y_coord, 2)],
                        'pass_end_location': None,
                        'pass_outcome': None,
                        'pass_goal_assist': 0,
                        'pass_type': None,
                        'shot_statsbomb_xg': np.nan,
                        'shot_outcome': None,
                        'play_pattern': None,
                    })

            for action_type, count in [
                ('Duel', num_duels),
                ('Interception', num_interceptions),
                ('Ball Recovery', num_recoveries),
            ]:
                for _ in range(count):
                    event_rows.append({
                        'match_id': match_id,
                        'team': team,
                        'type': action_type,
                        'location': [
                            round(float(np.random.uniform(0, 120)), 2),
                            round(float(np.random.uniform(0, 80)), 2),
                        ],
                        'pass_end_location': None,
                        'pass_outcome': None,
                        'pass_goal_assist': 0,
                        'pass_type': None,
                        'shot_statsbomb_xg': np.nan,
                        'shot_outcome': None,
                        'play_pattern': None,
                    })

    return (
        pd.DataFrame(event_rows)
        .sort_values(['match_id', 'team', 'type'])
        .reset_index(drop=True)
    )

matches_df = build_matches_df()
events_df = build_events_df(matches_df)

print(f'Matches: {len(matches_df)}')
print(f'Events: {len(events_df)}')
matches_df.tail(3)


## Performance Features: The Fundamentals

This section moves from direct output metrics, to venue-sensitive splits, and then to shot-quality features.

### Rolling Goal Averages


In [ ]:
all_teams_scores = pd.concat([
    matches_df[['match_date', 'home_team', 'home_score']].rename(
        columns={'home_team': 'team', 'home_score': 'goals'}
    ),
    matches_df[['match_date', 'away_team', 'away_score']].rename(
        columns={'away_team': 'team', 'away_score': 'goals'}
    ),
]).sort_values(by=['team', 'match_date'])

all_teams_scores['goals_rolling_avg_3'] = all_teams_scores.groupby('team')['goals'].transform(
    lambda values: values.rolling(window=3, min_periods=1).mean().shift(1)
)

for window in [5, 10]:
    all_teams_scores[f'goals_rolling_avg_{window}'] = (
        all_teams_scores.groupby('team')['goals'].transform(
            lambda values: values.rolling(window=window, min_periods=1).mean().shift(1)
        )
    )

all_teams_scores.head()


### Venue Splits: Home vs. Away


In [ ]:
home_stats = matches_df.groupby('home_team').agg({
    'home_score': ['mean', 'std'],
    'away_score': ['mean', 'std'],
}).reset_index()
home_stats.columns = [
    'team',
    'home_goals_avg',
    'home_goals_std',
    'home_conceded_avg',
    'home_conceded_std',
]

away_stats = matches_df.groupby('away_team').agg({
    'away_score': ['mean', 'std'],
    'home_score': ['mean', 'std'],
}).reset_index()
away_stats.columns = [
    'team',
    'away_goals_avg',
    'away_goals_std',
    'away_conceded_avg',
    'away_conceded_std',
]

team_stats = home_stats.merge(away_stats, on='team')
team_stats['home_advantage'] = team_stats['home_goals_avg'] - team_stats['away_goals_avg']
team_stats['home_defensive_advantage'] = (
    team_stats['away_conceded_avg'] - team_stats['home_conceded_avg']
)

team_stats.sort_values('home_advantage', ascending=False)


### Shot Quality: Evaluating Chance Creation


In [ ]:
shots = events_df[events_df['type'] == 'Shot'].copy()

xg_features = shots.groupby(['match_id', 'team']).agg({
    'shot_statsbomb_xg': ['sum', 'mean', 'count']
}).reset_index()
xg_features.columns = ['match_id', 'team', 'total_xg', 'avg_xg_per_shot', 'shots']

goals = shots[shots['shot_outcome'] == 'Goal'].groupby(['match_id', 'team']).size()
xg_features = xg_features.merge(
    goals.rename('goals'),
    on=['match_id', 'team'],
    how='left',
).fillna(0)
xg_features['xg_overperformance'] = xg_features['goals'] - xg_features['total_xg']
xg_features['shot_conversion_rate'] = xg_features['goals'] / (xg_features['shots'] + 1)

xg_features.head()


## Tactical Features: Describing How Teams Play

This section shifts from outcomes to style and pressure. Each block introduces one tactical metric family and then computes the corresponding match-level features.


### Passing and Possession


In [ ]:
passes = events_df[events_df['type'] == 'Pass'].copy()
passes['pass_success'] = passes['pass_outcome'].isna().astype(int)

passes['x_start'] = passes['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else 0)
passes['y_start'] = passes['location'].apply(lambda loc: loc[1] if isinstance(loc, list) else 0)
passes['x_end'] = passes['pass_end_location'].apply(
    lambda loc: loc[0] if isinstance(loc, list) else 0
)
passes['y_end'] = passes['pass_end_location'].apply(
    lambda loc: loc[1] if isinstance(loc, list) else 0
)

passes['pass_distance'] = np.sqrt(
    (passes['x_end'] - passes['x_start']) ** 2
    + (passes['y_end'] - passes['y_start']) ** 2
)
passes['progressive_pass'] = ((passes['x_end'] - passes['x_start']) > 10).astype(int)

pass_features = passes.groupby(['match_id', 'team']).agg({
    'pass_success': 'mean',
    'pass_distance': ['mean', 'std'],
    'progressive_pass': 'sum',
    'pass_goal_assist': 'sum',
}).reset_index()
pass_features.columns = [
    'match_id',
    'team',
    'pass_accuracy',
    'avg_pass_length',
    'std_pass_length',
    'progressive_passes',
    'assist_passes',
]

pass_features.head()


### Defensive Actions: Tackles, Interceptions, and Recoveries


In [ ]:
duels = events_df[events_df['type'] == 'Duel'].copy()
interceptions = events_df[events_df['type'] == 'Interception'].copy()
ball_recoveries = events_df[events_df['type'] == 'Ball Recovery'].copy()

teams = pd.unique(matches_df[['home_team', 'away_team']].values.ravel('K'))
defensive_frames = []

for team in teams:
    team_duels = duels[duels['team'] == team].groupby('match_id').size()
    team_interceptions = interceptions[interceptions['team'] == team].groupby('match_id').size()
    team_recoveries = ball_recoveries[ball_recoveries['team'] == team].groupby('match_id').size()

    match_ids = team_duels.index.union(team_interceptions.index).union(team_recoveries.index)
    defensive_frames.append(
        pd.DataFrame({
            'team': team,
            'match_id': match_ids,
            'duels_per_match': team_duels.reindex(match_ids, fill_value=0).values,
            'interceptions_per_match': (
                team_interceptions.reindex(match_ids, fill_value=0).values
            ),
            'recoveries_per_match': team_recoveries.reindex(match_ids, fill_value=0).values,
        })
    )

defensive_features = pd.concat(defensive_frames, ignore_index=True)
defensive_features['defensive_actions_total'] = (
    defensive_features['duels_per_match']
    + defensive_features['interceptions_per_match']
    + defensive_features['recoveries_per_match']
)

defensive_features.head()


### Pressing Intensity: PPDA


In [ ]:
ppda_features = []

for match_id in matches_df['match_id'].unique():
    match_events = events_df[events_df['match_id'] == match_id]
    home_team = matches_df[matches_df['match_id'] == match_id]['home_team'].iloc[0]
    away_team = matches_df[matches_df['match_id'] == match_id]['away_team'].iloc[0]

    away_passes_in_home_third = match_events[
        (match_events['team'] == away_team)
        & (match_events['type'] == 'Pass')
        & (match_events['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else 0) < 40)
    ].shape[0]

    home_passes_in_away_third = match_events[
        (match_events['team'] == home_team)
        & (match_events['type'] == 'Pass')
        & (match_events['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else 0) > 80)
    ].shape[0]

    home_def_actions = match_events[
        (match_events['team'] == home_team)
        & (match_events['type'].isin(['Duel', 'Interception', 'Ball Recovery']))
    ].shape[0]

    away_def_actions = match_events[
        (match_events['team'] == away_team)
        & (match_events['type'].isin(['Duel', 'Interception', 'Ball Recovery']))
    ].shape[0]

    ppda_features.append({
        'match_id': match_id,
        'home_team': home_team,
        'home_ppda': away_passes_in_home_third / (home_def_actions + 1),
        'away_team': away_team,
        'away_ppda': home_passes_in_away_third / (away_def_actions + 1),
    })

ppda_df = pd.DataFrame(ppda_features)
ppda_df.head()


### Set Pieces


In [ ]:
set_piece_map = {
    'Corner': 'Corner',
    'From Corner': 'Corner',
    'Free Kick': 'Free Kick',
    'From Free Kick': 'Free Kick',
    'Penalty': 'Penalty',
    'From Penalty': 'Penalty',
}

set_pieces = events_df[
    events_df['pass_type'].isin(['Corner', 'Free Kick'])
    | events_df['type'].isin(['Corner', 'Free Kick', 'Penalty'])
].copy()
set_pieces['set_piece_type'] = (
    set_pieces['pass_type'].fillna(set_pieces['type']).map(set_piece_map)
)

shots['set_piece_type'] = shots['play_pattern'].map(set_piece_map)
set_piece_shots = shots[
    shots['set_piece_type'].isin(set_pieces['set_piece_type'].dropna().unique())
].copy()

set_piece_features = set_piece_shots.groupby(['match_id', 'team']).agg({
    'shot_statsbomb_xg': 'sum',
    'shot_outcome': lambda values: (values == 'Goal').sum(),
}).reset_index()
set_piece_features.columns = [
    'match_id',
    'team',
    'set_piece_xg',
    'set_piece_goals',
]
set_piece_features['set_piece_conversion'] = (
    set_piece_features['set_piece_goals'] / (set_piece_features['set_piece_xg'] + 0.1)
)

set_piece_features.head()


## Contextual Features: Incorporating Match Conditions

### Rest and Scheduling


In [ ]:
matches_df['match_date'] = pd.to_datetime(matches_df['match_date'])
team_matches = pd.concat([
    matches_df[['match_date', 'match_id', 'home_team']].rename(columns={'home_team': 'team'}),
    matches_df[['match_date', 'match_id', 'away_team']].rename(columns={'away_team': 'team'}),
]).sort_values(['team', 'match_date'])

team_matches['days_rest'] = team_matches.groupby('team')['match_date'].diff().dt.days
team_matches['matches_played'] = team_matches.groupby('team').cumcount() + 1
team_matches['matches_in_last_14_days'] = (
    team_matches.groupby('team')
    .rolling('14D', on='match_date')['match_id']
    .count()
    .reset_index(level=0, drop=True)
    .to_numpy()
)

team_matches.head()


### Form and Momentum


In [ ]:
results = pd.concat([
    matches_df[['match_date', 'match_id', 'home_team', 'home_score', 'away_score']].assign(
        team=matches_df['home_team'],
        goals_for=matches_df['home_score'],
        goals_against=matches_df['away_score'],
    ),
    matches_df[['match_date', 'match_id', 'away_team', 'home_score', 'away_score']].assign(
        team=matches_df['away_team'],
        goals_for=matches_df['away_score'],
        goals_against=matches_df['home_score'],
    ),
])[
    ['match_date', 'match_id', 'team', 'goals_for', 'goals_against']
].sort_values(['team', 'match_date'])

results['result'] = np.where(
    results['goals_for'] > results['goals_against'],
    'W',
    np.where(results['goals_for'] < results['goals_against'], 'L', 'D'),
)
results['points'] = results['result'].map({'W': 3, 'D': 1, 'L': 0})
results['points_last_3'] = results.groupby('team')['points'].transform(
    lambda values: values.shift(1).rolling(3, min_periods=1).sum()
)
results['points_last_5'] = results.groupby('team')['points'].transform(
    lambda values: values.shift(1).rolling(5, min_periods=1).sum()
)

results['is_win'] = (results['result'] == 'W').astype(int)
results['win_pct_last_5'] = results.groupby('team')['is_win'].transform(
    lambda values: values.shift(1).rolling(5, min_periods=1).mean()
)

results.head()


## Ranking Features and Advanced Ranking Algorithms

### Colley Matrix


In [ ]:
def calculate_colley_ratings(matches_df):
    teams = pd.unique(matches_df[['home_team', 'away_team']].values.ravel('K'))
    team_to_idx = {team: idx for idx, team in enumerate(teams)}

    colley_matrix = np.zeros((len(teams), len(teams)))
    results_vector = np.zeros(len(teams))

    for _, row in matches_df.iterrows():
        home_idx = team_to_idx[row['home_team']]
        away_idx = team_to_idx[row['away_team']]

        colley_matrix[home_idx, home_idx] += 1
        colley_matrix[away_idx, away_idx] += 1
        colley_matrix[home_idx, away_idx] -= 1
        colley_matrix[away_idx, home_idx] -= 1

        if row['home_score'] > row['away_score']:
            results_vector[home_idx] += 1
            results_vector[away_idx] -= 1
        elif row['away_score'] > row['home_score']:
            results_vector[away_idx] += 1
            results_vector[home_idx] -= 1

    np.fill_diagonal(colley_matrix, colley_matrix.diagonal() + 2)
    ratings = np.linalg.solve(colley_matrix, results_vector)

    return pd.Series(ratings, index=teams, name='colley_rating').sort_values(
        ascending=False
    )

colley_df = calculate_colley_ratings(matches_df).rename_axis('team').reset_index()
colley_df.head()


### PageRank


In [ ]:
def calculate_pagerank_ratings(matches_df, damping=0.85):
    teams = pd.unique(matches_df[['home_team', 'away_team']].values.ravel('K'))
    graph = nx.DiGraph()
    graph.add_nodes_from(teams)

    for _, row in matches_df.iterrows():
        home_team = row['home_team']
        away_team = row['away_team']
        home_score = row['home_score']
        away_score = row['away_score']

        if home_score > away_score:
            graph.add_edge(away_team, home_team, weight=home_score - away_score)
        elif away_score > home_score:
            graph.add_edge(home_team, away_team, weight=away_score - home_score)
        else:
            graph.add_edge(home_team, away_team, weight=0.5)
            graph.add_edge(away_team, home_team, weight=0.5)

    return pd.Series(
        nx.pagerank(graph, weight='weight', alpha=damping),
        name='pagerank',
    ).sort_values(ascending=False)

pagerank_df = calculate_pagerank_ratings(matches_df).rename_axis('team').reset_index()
pagerank_df.head()


### Elo Ratings


In [ ]:
def calculate_elo_ratings(matches_df, K=30, initial_rating=1500, home_advantage=0):
    """Calculate Elo ratings for all teams."""
    ratings = {}
    rating_history = []

    for _, match in matches_df.sort_values('match_date').iterrows():
        home_team = match['home_team']
        away_team = match['away_team']

        if home_team not in ratings:
            ratings[home_team] = initial_rating
        if away_team not in ratings:
            ratings[away_team] = initial_rating

        home_rating_adjusted = ratings[home_team] + home_advantage
        expected_home = 1 / (1 + 10 ** ((ratings[away_team] - home_rating_adjusted) / 400))
        expected_away = 1 - expected_home

        if match['home_score'] > match['away_score']:
            actual_home, actual_away = 1, 0
        elif match['home_score'] < match['away_score']:
            actual_home, actual_away = 0, 1
        else:
            actual_home, actual_away = 0.5, 0.5

        ratings[home_team] += K * (actual_home - expected_home)
        ratings[away_team] += K * (actual_away - expected_away)

        rating_history.append({
            'match_date': match['match_date'],
            'home_team': home_team,
            'away_team': away_team,
            'home_rating': ratings[home_team],
            'away_rating': ratings[away_team],
        })

    final_ratings = pd.Series(ratings, name='elo_rating').sort_values(ascending=False)
    return pd.DataFrame(rating_history), final_ratings

elo_history, final_elo = calculate_elo_ratings(matches_df, K=30)
elo_df = final_elo.rename_axis('team').reset_index()
elo_df.head()


### Comparing Methods


In [ ]:
all_rankings = colley_df.merge(pagerank_df, on='team').merge(elo_df, on='team')

for column_name in ['colley_rating', 'pagerank', 'elo_rating']:
    all_rankings[f'{column_name}_norm'] = (
        (all_rankings[column_name] - all_rankings[column_name].min())
        / (all_rankings[column_name].max() - all_rankings[column_name].min())
    )

print('Correlation between ranking methods:')
print(all_rankings[['colley_rating_norm', 'pagerank_norm', 'elo_rating_norm']].corr())

all_rankings['colley_elo_diff'] = abs(
    all_rankings['colley_rating_norm'] - all_rankings['elo_rating_norm']
)
print() 
print('Teams with biggest disagreement between Colley and Elo:')
print(
    all_rankings.nlargest(5, 'colley_elo_diff')[['team', 'colley_rating', 'elo_rating']]
)


## Feature Engineering Pipeline: From Data to Model Input


In [ ]:
def engineer_match_features(matches_df, events_df, match_id, home_team, away_team):
    """Engineer features for a single match prediction."""

    matches_df = matches_df.copy()
    matches_df['match_date'] = pd.to_datetime(matches_df['match_date'])
    match_date = matches_df[matches_df['match_id'] == match_id]['match_date'].iloc[0]
    past_matches = matches_df[matches_df['match_date'] < match_date].sort_values('match_date')
    past_events = events_df[events_df['match_id'].isin(past_matches['match_id'])]

    team_history = pd.concat([
        past_matches[['match_date', 'home_team', 'home_score', 'away_score']]
        .rename(columns={
            'home_team': 'team',
            'home_score': 'goals_for',
            'away_score': 'goals_against',
        })
        .assign(venue='home'),
        past_matches[['match_date', 'away_team', 'away_score', 'home_score']]
        .rename(columns={
            'away_team': 'team',
            'away_score': 'goals_for',
            'home_score': 'goals_against',
        })
        .assign(venue='away'),
    ]).sort_values(['team', 'match_date'])

    features = {}

    home_recent = team_history[team_history['team'] == home_team].tail(5)
    away_recent = team_history[team_history['team'] == away_team].tail(5)
    features['home_goals_avg_5'] = home_recent['goals_for'].mean()
    features['away_goals_avg_5'] = away_recent['goals_for'].mean()

    home_at_home = team_history[
        (team_history['team'] == home_team) & (team_history['venue'] == 'home')
    ]
    away_at_away = team_history[
        (team_history['team'] == away_team) & (team_history['venue'] == 'away')
    ]
    features['home_home_goals_avg'] = home_at_home['goals_for'].mean()
    features['away_away_goals_avg'] = away_at_away['goals_for'].mean()

    home_shots = past_events[
        (past_events['team'] == home_team) & (past_events['type'] == 'Shot')
    ]
    away_shots = past_events[
        (past_events['team'] == away_team) & (past_events['type'] == 'Shot')
    ]
    features['home_avg_xg'] = home_shots['shot_statsbomb_xg'].mean()
    features['away_avg_xg'] = away_shots['shot_statsbomb_xg'].mean()

    colley = calculate_colley_ratings(past_matches)
    features['home_colley'] = colley.get(home_team, 0.5)
    features['away_colley'] = colley.get(away_team, 0.5)
    features['colley_diff'] = features['home_colley'] - features['away_colley']

    _, elo = calculate_elo_ratings(past_matches)
    features['home_elo'] = elo.get(home_team, 1500)
    features['away_elo'] = elo.get(away_team, 1500)
    features['elo_diff'] = features['home_elo'] - features['away_elo']

    for side, team in [('home', home_team), ('away', away_team)]:
        team_prior_matches = past_matches[
            (past_matches['home_team'] == team) | (past_matches['away_team'] == team)
        ].sort_values('match_date')

        if not team_prior_matches.empty:
            last_match_date = team_prior_matches.iloc[-1]['match_date']
            features[f'{side}_days_rest'] = (match_date - last_match_date).days

    return features


## Case Study: Predicting the 2019 Women’s World Cup Final


In [ ]:
final_match_id = 69321
features = engineer_match_features(
    matches_df,
    events_df,
    final_match_id,
    home_team="United States Women's",
    away_team="Netherlands Women's",
)

print('=== USA vs Netherlands Final: Feature Analysis ===')
print()
print('Form (goals rolling avg, last 5 matches):')
print(f"  USA: {features['home_goals_avg_5']:.2f} goals/game")
print(f"  Netherlands: {features['away_goals_avg_5']:.2f} goals/game")

print() 
print('Home/Away Splits:')
print(f"  USA at home: {features['home_home_goals_avg']:.2f} goals/game")
print(f"  Netherlands away: {features['away_away_goals_avg']:.2f} goals/game")

print() 
print('Rankings:')
print(f"  USA Colley: {features['home_colley']:.3f}")
print(f"  Netherlands Colley: {features['away_colley']:.3f}")
print(f"  Colley difference: {features['colley_diff']:.3f}")
print(f"  Elo difference: {features['elo_diff']:.0f} points")

print() 
print('Shot Quality (avg xG per shot):')
print(f"  USA: {features['home_avg_xg']:.3f}")
print(f"  Netherlands: {features['away_avg_xg']:.3f}")

print() 
print('Rest:')
print(f"  USA days rest: {features.get('home_days_rest', 'N/A')}")
print(f"  Netherlands days rest: {features.get('away_days_rest', 'N/A')}")

print() 
print('=== Interpretation ===')
print('USA had better form, higher rankings, and stronger shot quality signals.')
print('The engineered features point toward a USA win, which matches the 2-0 result.')
